# Act 3: Deep Field Galaxy Classification & Downlink Optimization

Distant galaxy surveys reveal diverse morphologies, including smooth ellipticals, spiral disks, edge-on dust lanes, and colliding mergers. Deep convolutional neural networks learn spatial filters that identify structural features like spiral arms and tidal tails directly from pixel arrays. Because downlink bandwidth is limited to 20 images, we must prioritize downlinking the most scientifically valuable galaxies. We balance classification probability against scientific utility weights to maximize discovery value.

In [ ]:
# Setup and package installation
REPO_URL = "https://github.com/placeholder/meridian"

import os
import subprocess
import sys
from pathlib import Path

# Enable FAST mode by default to respect compute budgets (set to "0" to disable)
os.environ.setdefault("MERIDIAN_FAST", "1")

try:
    import meridian
except ImportError:
    if "google.colab" in sys.modules:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"git+{REPO_URL}"], check=True)
        import meridian
    else:
        repo_dir = Path.cwd()
        while repo_dir.parent != repo_dir and not (repo_dir / "meridian").exists():
            repo_dir = repo_dir.parent
        if (repo_dir / "meridian").exists() and str(repo_dir) not in sys.path:
            sys.path.insert(0, str(repo_dir))
        import meridian

fast_str = "ON" if os.environ.get("MERIDIAN_FAST") == "1" else "OFF"
backend_str = "torch" if getattr(meridian, "HAS_TORCH", False) else "sklearn"
print(f"Meridian v{meridian.__version__} | FAST mode: {fast_str} | Act 3 backend: {backend_str}")


In [ ]:
import os
TEAM_ID = os.environ.get("MERIDIAN_TEAM_ID", "your-team-name")

from meridian.state import load_state
from meridian.ui import briefing, dashboard, debrief
state = load_state(TEAM_ID)


In [ ]:
briefing(3)

### Loading the Sky Survey

The survey contains 48x48 pixel images categorized into 4 astronomical classes. Merger systems and spiral galaxies carry the highest scientific value weights for astrophysics research. Your model must accurately classify morphology despite faint signal levels and read noise.

In [ ]:
from meridian.act3_galaxy import (
    build_survey, train, select_downlink, score,
    show_samples, plot_confusion, show_downlink, human_vs_ai, reveal
)
survey = build_survey(TEAM_ID)
print(f'Galaxy survey loaded: {len(survey.train_images)} training images, {len(survey.stream_images)} sky stream images.')

### Human vs AI Warm-up

Can you visually distinguish between edge-on disks, smooth ellipticals, and colliding merger galaxies? Test your morphological classification skills on the four unlabeled sample images below. Automated classifiers evaluate subtle pixel gradients and asymmetry metrics.

In [ ]:
# Challenge: Classify 4 galaxy morphology samples
challenge = human_vs_ai(survey, n=4)

In [ ]:
# YOUR GUESSES: Replace None with class indices (0: Elliptical, 1: Spiral, 2: Edge-on, 3: Merger).
# e.g. guesses = [0, 1, 2, 3]. Unedited run will treat None as skipped.
guesses = [None, None, None, None]
reveal(guesses=guesses, challenge=challenge)

### Inspecting Morphological Classes

Let us inspect reference examples from each morphological class in the training set. Notice how spirals display curved arm structures while mergers exhibit irregular asymmetric tidal debris. These spatial patterns guide the convolutional network during training.

In [ ]:
# Inspect sample galaxy morphologies
import matplotlib.pyplot as plt
show_samples(survey, n_per_class=3)
plt.show()

### YOUR TURN: Train Classifier and Select Downlink Strategy

Configure training epochs, data augmentation, and downlink selection strategy. The 'expected_value' strategy weights class probabilities by scientific value (mergers=5, spirals=3). Data augmentation rotates and flips training images to improve generalization.

In [ ]:
# YOUR TURN: Adjust Act 3 Galaxy Classifier and Downlink hyperparameters
knobs = {
    "epochs": 5,
    "augment": True,
    "class_weighting": True,
    "strategy": "expected_value",
}

In [ ]:
# Train image classifier and optimize scientific downlink
model = train(survey, **knobs)
selected = select_downlink(model, survey, strategy=knobs["strategy"])
act3_score = score(state, survey, model, selected)
print(f'Act 3 Score: {state.act3.score:.1f}/200 | Accuracy: {state.act3.accuracy:.1%} | Value Ratio: {state.act3.value_ratio:.1%}')

In [ ]:
# Display confusion matrix and prioritized downlink stream
plot_confusion(model, survey)
plt.show()
show_downlink(survey, selected, model)
plt.show()

In [ ]:
debrief(state, 3)

In [ ]:
state.save()
dashboard(state)